# dbt Data Contracts & Model Versioning

A live demo using **dbt-core + DuckDB**. No warehouse required — everything runs locally.

---

## What we'll cover

| Chapter | Topic |
|---------|-------|
| 1 | Raw data — seed tables as the source of truth |
| 2 | Staging layer — transformations without contracts |
| 3 | Data contracts — locking the gold layer |
| 4 | **Contract violation** — what dbt catches and why it matters |
| 5 | **Model versioning** — evolving contracts without breaking consumers |

In [ ]:
import subprocess
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_DIR = Path.cwd()
DB_PATH = PROJECT_DIR / "demo.duckdb"

print(f"Project : {PROJECT_DIR}")
print(f"Database: {DB_PATH}")

In [ ]:
import shutil

# ── Clean slate ────────────────────────────────────────────────────────────
# Remove all dbt artifacts and the database so every run starts fresh.
artifacts = [
    PROJECT_DIR / "demo.duckdb",
    PROJECT_DIR / "demo.duckdb.wal",
    PROJECT_DIR / "target",
    PROJECT_DIR / "logs",
    PROJECT_DIR / "dbt_packages",
]

removed = []
for path in artifacts:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        removed.append(path.name)

if removed:
    print(f"Removed: {', '.join(removed)}")
else:
    print("Nothing to clean.")
print("Clean slate ready ✓")

## Setup: Build the dbt Pipeline

All artifacts from previous runs (`demo.duckdb`, `target/`, `logs/`) have been removed. Now load the seed data and build all models from scratch.

In [ ]:
def run_dbt(args):
    """Run a dbt command, print concise output, return the result."""
    cmd = ["dbt"] + args + ["--profiles-dir", str(PROJECT_DIR)]
    print(f"$ dbt {' '.join(args)}")
    print("─" * 70)
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

    if result.returncode == 0:
        for line in result.stdout.splitlines():
            # Show: model counts, per-model OK lines, timing summary, done
            if any(kw in line for kw in ["Found", " OK ", "Finished running", "Done."]):
                print(line)
        print("\n✅  Completed successfully")
    else:
        print(result.stdout)
        if result.stderr.strip():
            print(result.stderr)

    return result


# Step 1: load CSVs into main_raw schema
run_dbt(["seed"])
print()

# Step 2: build all models (skip the intentionally broken one — that comes in Chapter 4)
run_dbt(["run", "--exclude", "fct_orders_broken"])

In [ ]:
# Open a connection to the database (dbt has already written everything)
con = duckdb.connect(str(DB_PATH))

print("Connected to demo.duckdb ✓\n")
print("All tables in the database:")
con.sql("SHOW ALL TABLES").df()[["database", "schema", "name", "column_names"]]

## Pipeline Architecture

```
┌──────────────────┐   ┌───────────────────────┐   ┌───────────────────────┐   ┌──────────────────────┐
│   main_raw       │   │   main_staging        │   │   main_gold           │   │   main_serving       │
│──────────────────│   │───────────────────────│   │───────────────────────│   │──────────────────────│
│ raw_orders       │──▶│ stg_orders  (view)    │──▶│ fct_orders_v1  table  │   │ fct_orders  (view)   │
│ raw_customers    │──▶│ stg_customers (view)  │──▶│ fct_orders_v2  table  │──▶│   → fct_orders_v2    │
│ raw_products     │──▶│ stg_products  (view)  │──▶│ dim_customers  table  │   │                      │
│                  │   │                       │   │                       │   │                      │
│ Seeds (CSV)      │   │ No contracts          │   │ ✓ Contracts enforced  │   │ Public interface     │
└──────────────────┘   └───────────────────────┘   └───────────────────────┘   └──────────────────────┘
```

**Contracts lock the gold layer** — stable schema guarantee for all consumers.
**The serving layer** exposes a single versioned view: consumers always query `fct_orders`, never the versioned tables directly.

---
## Chapter 1: Raw Data

Three seed tables loaded from CSV files. This is our source of truth — no transformations, no guarantees, just raw records.

In [ ]:
for table, label in [
    ("main_raw.raw_customers", "raw_customers — 6 rows"),
    ("main_raw.raw_products",  "raw_products — 5 rows"),
    ("main_raw.raw_orders",    "raw_orders — 20 rows"),
]:
    print(f"\n{'━' * 60}")
    print(f"  {label}")
    print(f"{'━' * 60}")
    display(con.sql(f"SELECT * FROM {table}").df())

---
## Chapter 2: The Staging Layer — Transformations, No Contracts

Staging models cast raw types, rename columns, and clean the data. They are **intentionally uncontracted** — this is the fast-iteration zone.

```sql
-- models/staging/stg_orders.sql
select
    cast(order_id    as bigint)        as order_id,
    cast(customer_id as bigint)        as customer_id,
    cast(product_id  as bigint)        as product_id,
    cast(amount      as decimal(10,2)) as amount,
    cast(status      as varchar)       as status,
    cast(created_at  as date)          as created_at
from {{ ref('raw_orders') }}
```

No `contract: enforced: true` here. If a developer renames `order_id` to `id` in this model, dbt will happily run it. The problem only surfaces downstream.

In [ ]:
print("stg_orders — column types (no contract, no constraints):")
display(con.sql("DESCRIBE main_staging.stg_orders").df())

print("\nDatabase-level constraints on stg_orders:")
constraints = con.sql("""
    SELECT constraint_type, constraint_column_names
    FROM duckdb_constraints()
    WHERE table_name = 'stg_orders'
""").df()
if constraints.empty:
    print("  (none — staging is a view with no enforced constraints)")
else:
    display(constraints)

print("\nFirst 5 rows:")
display(con.sql("SELECT * FROM main_staging.stg_orders LIMIT 5").df())

---
## Chapter 3: Data Contracts — Locking the Gold Layer

At the gold layer, models declare a **data contract**: an explicit promise about column names, types, and constraints.

```yaml
# models/gold/_dim_customers.yml
models:
  - name: dim_customers
    config:
      contract:
        enforced: true          # ← turns on contract enforcement
    columns:
      - name: customer_id
        data_type: bigint       # ← declared type
        constraints:
          - type: not_null      # ← real DB-level NOT NULL
          - type: unique        # ← real DB-level UNIQUE
      - name: name
        data_type: varchar
      - name: email
        data_type: varchar
      - name: country
        data_type: varchar
```

When `enforced: true`, dbt generates a `CREATE TABLE` with **explicit column types and constraints**, then inserts the model's output. If the output doesn't match the declared schema → **build fails immediately**.

In [ ]:
print("dim_customers — column schema (contract enforced):")
display(con.sql("DESCRIBE main_gold.dim_customers").df())

print("\nDatabase-level constraints applied to dim_customers:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'dim_customers'
    """).df()
)

In [ ]:
print("fct_orders_v2 — column schema (contract enforced):")
display(con.sql("DESCRIBE main_gold.fct_orders_v2").df())

print("\nDatabase-level constraints applied to fct_orders_v2:")
display(
    con.sql("""
        SELECT constraint_type, constraint_column_names
        FROM duckdb_constraints()
        WHERE table_name = 'fct_orders_v2'
    """).df()
)

In [ ]:
print("dim_customers data (6 rows):")
display(con.sql("SELECT * FROM main_gold.dim_customers").df())

print("\nfct_orders_v2 data (first 5 rows):")
display(con.sql("SELECT * FROM main_gold.fct_orders_v2 LIMIT 5").df())

### Under the hood: the generated `CREATE TABLE`

When `contract: enforced: true`, dbt generates a `CREATE TABLE` statement with **explicit types and constraints** baked in, then inserts the model's output into it. Here's the actual DDL dbt compiled for `fct_orders_v2`:

In [ ]:
import re

ddl_path = PROJECT_DIR / "target/run/dbt_contracts_demo/models/gold/fct_orders_v2.sql"
ddl_text = ddl_path.read_text()

# Extract just the CREATE TABLE...;  block (everything up to the first semicolon)
start = ddl_text.lower().find("create")
end = ddl_text.find(";", start) + 1
create_stmt = ddl_text[start:end]

# Clean up Jinja whitespace artifacts
create_stmt = re.sub(r"[ \t]+\n", "\n", create_stmt)
create_stmt = re.sub(r"\n{3,}", "\n\n", create_stmt)
create_stmt = re.sub(r"  +", " ", create_stmt)
create_stmt = create_stmt.strip()

print(create_stmt)
print()
print("─" * 70)
print("Every column type and constraint comes directly from the YAML contract.")
print("dbt generated this DDL — the developer never writes it by hand.")

---
## Chapter 4: Contract Violation

A developer renames `order_id` → `id` in `fct_orders`. The SQL is valid. Without contracts, this deploys silently and breaks every downstream consumer.

```sql
-- models/serving/fct_orders_broken.sql
select
    o.order_id  as id,          -- renamed: contract expects 'order_id'
    o.customer_id,
    o.product_id,
    o.amount,
    o.status,
    o.created_at    as order_date,
    p.category      as product_category
from orders o
left join products p on o.product_id = p.product_id
```

The contract declares `order_id bigint not null`. Let's run it.

In [ ]:
print("Running the broken model against its contract...\n")

# DuckDB allows only one writer at a time.
# We close our read connection so dbt can acquire the write lock.
con.close()

cmd = [
    "dbt", "run",
    "--select", "fct_orders_broken",
    "--profiles-dir", str(PROJECT_DIR),
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_DIR))

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

if result.returncode != 0:
    print("\n" + "═" * 70)
    print("  ❌  dbt REFUSED to build this model")
    print("═" * 70)

# Reopen for subsequent analysis cells
con = duckdb.connect(str(DB_PATH))

### What dbt caught

dbt shows exactly what differs between the model output and the contract:

```
| column_name | definition_type | contract_type | mismatch_reason       |
| ----------- | --------------- | ------------- | --------------------- |
| id          | BIGINT          |               | missing in contract   |
| order_id    |                 | BIGINT        | missing in definition |
```

**Nothing was written to the database.** The build failed at compile time — caught before any SQL ran, not at 2 am.

---
## Chapter 5: Model Versioning — Evolving Contracts Without Breaking Consumers

Contracts prevent silent breakage, but what if you *need* to change the schema? Adding `product_category` is backward-incompatible. The answer: **model versioning**.

```yaml
# models/serving/_fct_orders.yml
models:
  - name: fct_orders
    latest_version: 2
    config:
      contract:
        enforced: true

    columns:
      - name: order_id
        data_type: bigint
        constraints: [not_null]
      - name: customer_id
        data_type: bigint
      # ... (amount, status, order_date)

    versions:
      - v: 1
      - v: 2
        defined_in: fct_orders_v2
        columns:
          - include: all
          - name: product_category
            data_type: varchar
```

Both versions are built simultaneously. Consumers choose which to reference:

| Consumer code | Resolves to | Columns |
|---|---|---|
| `ref('fct_orders')` | **v2** (latest) | 7 — incl. `product_category` |
| `ref('fct_orders', version=1)` | v1 | 6 — stable |

In [ ]:
v1 = con.sql("SELECT * FROM main_gold.fct_orders_v1 LIMIT 5").df()
v2 = con.sql("SELECT * FROM main_gold.fct_orders_v2 LIMIT 5").df()

print(f"fct_orders  v1  — {len(v1.columns)} columns: {list(v1.columns)}")
display(v1)

print(f"\nfct_orders  v2  — {len(v2.columns)} columns: {list(v2.columns)}")
display(v2)

new_cols = [c for c in v2.columns if c not in v1.columns]
print(f"\nNew in v2: {new_cols}")

In [ ]:
# Gold layer: both versioned tables coexist
rows = []
for tbl, version, note in [
    ("fct_orders_v1", "v1", "baseline — 6 cols, stable"),
    ("fct_orders_v2", "v2 (latest)", "extended — 7 cols, adds product_category"),
    ("dim_customers",  "—", "dimension table"),
]:
    row_count = con.sql(f"SELECT COUNT(*) FROM main_gold.{tbl}").fetchone()[0]
    cols = con.sql(
        f"SELECT column_name FROM information_schema.columns "
        f"WHERE table_name = '{tbl}' AND table_schema = 'main_gold' ORDER BY ordinal_position"
    ).df()["column_name"].tolist()
    rows.append({
        "table": f"main_gold.{tbl}",
        "version": version,
        "rows": row_count,
        "columns": len(cols),
        "column names": ", ".join(cols),
    })

display(pd.DataFrame(rows))

In [ ]:
# The serving view — what consumers actually query
print("main_serving.fct_orders — the public interface:")
display(con.sql("SELECT * FROM main_serving.fct_orders LIMIT 5").df())

print("\nView definition (what dbt generated):")
view_def = con.sql("""
    SELECT view_definition
    FROM duckdb_views()
    WHERE view_name = 'fct_orders' AND schema_name = 'main_serving'
""").fetchone()[0]
print(view_def)

### Migration path

The cutover is a single line change in `models/serving/fct_orders.sql` — updating the `ref()` to point to the new version.

**Before cutover** — serving view points to v1:

| Layer | Model | Type | Note |
|---|---|---|---|
| `main_gold` | `fct_orders_v1` | Table | Production |
| `main_gold` | `fct_orders_v2` | Table | QA / staging |
| `main_serving` | `fct_orders` | **View** | `→ ref('fct_orders_v1')` |

**After cutover** — one-line change in `fct_orders.sql`; v1 gets a deprecation date:

| Layer | Model | Type | Note |
|---|---|---|---|
| `main_gold` | `fct_orders_v1` | Table | Deprecated — grace period |
| `main_gold` | `fct_orders_v2` | Table | Production |
| `main_serving` | `fct_orders` | **View** | `→ ref('fct_orders_v2')` ✓ |

Consumers always query `main_serving.fct_orders` — they never need to change their code.

---
## Summary

| Feature | Key takeaway |
|---------|-------------|
| **`contract: enforced: true`** | Generates a typed `CREATE TABLE` — column names, types, and constraints declared in YAML, enforced by the DB |
| **Column constraints** | `not_null`, `unique`, `primary_key` at the database level — not just dbt tests |
| **Contract violation** | Build fails **before any data is written** if model output doesn't match the declared schema |
| **`latest_version: N`** | `ref('fct_orders')` auto-resolves to the current latest version |
| **Version pinning** | `ref('fct_orders', version=1)` — legacy consumers never break during a schema migration |

---

*Stack: dbt-core · dbt-duckdb · DuckDB*